# 1. What are Duplicate Records?

### Concept & Definition
Duplicate records occur when multiple rows in a dataset contain identical or near-identical information across all or a subset of features. Duplicates typically enter systems through repeated database entries, batch join errors, system retries, or merging un-deduplicated logs.

### Real-World / Business Example
A customer clicks "Submit" twice on a web sign-up form, resulting in two identical user records created within milliseconds of each other.

### ML Impact
Duplicate records skew evaluation metrics, cause artificial data leakage when identical rows fall into both Train and Test splits, and overweight redundant observations during model fitting.

In [1]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Customer_Data.csv")

print("=== Raw Dataset Summary ===")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {df.shape[1]}")

=== Raw Dataset Summary ===
Total Rows: 1010
Total Columns: 9


# 2. Exact Duplicates

### Concept & Definition
An **Exact Duplicate** is a record where every single column value across the entire row is 100% identical to another row in the dataset.

### Real-World / Business Example
A database back-up script runs twice, duplicating every transaction row without altering primary keys or timestamps.

### Why It Solves Problems
Removing exact duplicates reduces memory footprint and eliminates redundant parameter updates during gradient descent.

In [2]:
# Detecting Exact Duplicates
exact_duplicates_count = df.duplicated(subset=None, keep="first").sum()

print(f"Total Exact Duplicate Rows Found: {exact_duplicates_count}")

# Inspecting sample exact duplicate rows
if exact_duplicates_count > 0:
    print("\nSample Exact Duplicate Records:")
    display(df[df.duplicated(subset=None, keep=False)].head(6))

Total Exact Duplicate Rows Found: 10

Sample Exact Duplicate Records:


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No
3,CUST-1003,45.0,Female,10,$56.95,7529.311738,Month-to-month,Bank transfer,Yes
4,CUST-1004,25.0,M,3,$29.85,225.003059,One year,Mailed check,Yes
5,CUST-1005,25.0,Female,8,$29.85,4229.805118,month to month,Electronic check,No


# 3. Partial Duplicates

### Concept & Definition
A **Partial Duplicate** occurs when two or more records match identically on a specific subset of key columns (e.g., `CustomerID` or `Email`), even if other columns (like `CreatedDate` or `SessionID`) differ slightly.

### Real-World / Business Example
A customer updates their `MonthlyCharges` plan, resulting in two rows with the same `CustomerID` but different subscription values or update timestamps.

### Why It Solves Problems
Identifies entity duplication where the same customer exists multiple times due to profile updates or multiple system activity logs.

In [3]:
# Detecting Partial Duplicates based on Primary Business Identifier (CustomerID)
partial_duplicates = df.duplicated(subset=["CustomerID"], keep=False)

print(f"Total Partial Duplicates (Matching CustomerID): {partial_duplicates.sum()}")

if partial_duplicates.sum() > 0:
    print("\nSample Partial Duplicate Records:")
    display(df[partial_duplicates].sort_values(by="CustomerID").head(6))

Total Partial Duplicates (Matching CustomerID): 20

Sample Partial Duplicate Records:


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1000,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
1001,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No
1002,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No


# 4. Identifying Duplicates using duplicated()

### Concept & Definition
Pandas provides the `.duplicated()` method to return a boolean mask flagging duplicate rows.

### Key Parameters:
- `subset`: List of column names to consider for identifying duplicates.
- `keep='first'`: Flags duplicates as `True` except for the first occurrence.
- `keep='last'`: Flags duplicates as `True` except for the last occurrence.
- `keep=False`: Flags **all** occurrences of duplicate rows as `True`.

### ML Impact
Using `keep=False` allows complete auditing of duplicate clusters before making deletion decisions.

In [4]:
# Comparing keep='first', keep='last', and keep=False
dup_first = df.duplicated(keep="first").sum()
dup_last = df.duplicated(keep="last").sum()
dup_all = df.duplicated(keep=False).sum()

print(f"Duplicates marked with keep='first': {dup_first}")
print(f"Duplicates marked with keep='last': {dup_last}")
print(f"Total rows involved in duplicate pairs (keep=False): {dup_all}")

Duplicates marked with keep='first': 10
Duplicates marked with keep='last': 10
Total rows involved in duplicate pairs (keep=False): 20


# 5. Handling Duplicates using drop_duplicates()

### Concept & Definition
`drop_duplicates()` removes duplicate rows from a DataFrame based on specified columns and retention strategies.

### When to Use:
- Removing exact duplicates to clean raw data ingestions.

### When NOT to Use:
- When duplicates represent valid recurring occurrences (e.g., repeated valid transactions in retail logs).

In [5]:
# Executing exact duplicate removal
df_deduplicated = df.drop_duplicates(subset=None, keep="first")

print(f"Original Row Count: {len(df)}")
print(f"Deduplicated Row Count: {len(df_deduplicated)}")
print(f"Rows Removed: {len(df) - len(df_deduplicated)}")

Original Row Count: 1010
Deduplicated Row Count: 1000
Rows Removed: 10


# 6. Duplicate Detection Based on Selected Columns

### Concept & Definition
In business databases, primary keys or core composite keys (e.g., `["CustomerID", "ContractType"]`) define uniqueness, even if secondary metadata columns differ.

### Real-World / Business Example
Filtering duplicate survey submissions where a user with the same `CustomerID` submitted two survey responses on the same day.

### ML Impact
Ensures that each unique real-world entity is represented exactly once in the feature matrix.

In [6]:
# Deduplicating based on specific business columns
df_custom_dedup = df.drop_duplicates(subset=["CustomerID"], keep="last")

print(f"Shape after keeping LAST record per CustomerID: {df_custom_dedup.shape}")

Shape after keeping LAST record per CustomerID: (1000, 9)


# 7. Business Rules for Duplicate Removal

### Concept & Definition
Duplicate removal should never be an automated blind operation; it must follow domain-specific business rules.

### Common Business Rules:
1. **Keep Most Recent (`keep='last'`):** Preserve the most recent updated profile entry.
2. **Keep Most Complete:** Choose the record with fewer missing values (`NaN`).
3. **Aggregate / Merge:** Sum or average numerical attributes across duplicates before removal.

### Real-World / Business Example
In financial ledger logs, two identical $100 withdrawals might represent two real, legitimate transactions made seconds apart rather than a system glitch.

In [7]:
# Custom Business Logic: Select the record with the fewest null values among duplicates
def deduplicate_by_completeness(dataframe, key_column):
    # Sort by number of non-null values descending
    dataframe["non_null_count"] = dataframe.notnull().sum(axis=1)
    sorted_df = dataframe.sort_values(
        by=[key_column, "non_null_count"], ascending=[True, False]
    )

    # Drop duplicates keeping the record with the highest non-null count
    clean_df = sorted_df.drop_duplicates(subset=[key_column], keep="first")
    return clean_df.drop(columns=["non_null_count"])


df_business_clean = deduplicate_by_completeness(df, key_column="CustomerID")
print(
    f"Business Deduplication Completed. Remaining Rows: {len(df_business_clean)}"
)

Business Deduplication Completed. Remaining Rows: 1000


# 8. Risks of Inappropriate Duplicate Removal

### Concept & Definition
Blindly deleting duplicate rows without understanding the underlying domain logic can destroy genuine frequency signals and introduce sample bias.

### Why Blindly Deleting Duplicates Can Be Dangerous:
1. **Destroying Class Distributions:** In fraud detection or rare event prediction, duplicate entries may reflect repeated attacks or valid high-frequency activities. Removing them undercounts critical risk factors.
2. **Loss of Frequency Weights:** In transaction logs, two rows with identical values might represent genuine repeat purchases.
3. **Improper Time-Series Filtering:** In sequential data, identical readings at consecutive time steps indicate steady-state conditions; dropping them breaks time-series continuity.

In [8]:
# Demonstrating impact of deduplication on target class balance
print("Target Class Distribution Before Deduplication:")
print(df["Churn"].value_counts(normalize=True).round(4) * 100)

print("\nTarget Class Distribution After Deduplication:")
print(df_deduplicated["Churn"].value_counts(normalize=True).round(4) * 100)

Target Class Distribution Before Deduplication:
Churn
No     69.6
Yes    30.4
Name: proportion, dtype: float64

Target Class Distribution After Deduplication:
Churn
No     69.6
Yes    30.4
Name: proportion, dtype: float64


# 9. Notebook Summary & Clean Checkpoint

### Summary Checklist:
1. Audited total and partial duplicate records using `.duplicated()`.
2. Applied business-rule deduplication retaining the most complete customer profile.
3. Verified target label balance stability post-deduplication.

In [9]:
# Save clean deduplicated checkpoint
df_clean_dedup = df.drop_duplicates(subset=["CustomerID"], keep="first").copy()

print(f"Final Deduplicated Dataset Shape: {df_clean_dedup.shape}")
print("Notebook 04 execution completed successfully!")

Final Deduplicated Dataset Shape: (1000, 10)
Notebook 04 execution completed successfully!
